# Проверка демонстрационного набора Green Taxi

Этот ноутбук показывает минимальную проверку документации и файла для ЛР № 1. Он читает raw-файлы без преобразования и не является пайплайном загрузки или очистки.

Перед запуском откройте: [паспорт raw-файлов](raw/README.md), [Project Brief](Project_Brief_демонстрационный_пример.md) и [словарь Green Taxi](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_green.pdf).

В ноутбуке используются `duckdb` и `pandas`. Если пакеты не установлены, выполните в терминале `pip install duckdb pandas`.

Функция `query_df(...)` выполняет SQL-запрос и возвращает результат как таблицу `DataFrame`, которую Jupyter отображает в удобном виде.

In [ ]:
from pathlib import Path
import hashlib
import duckdb

RAW_DIR = Path('raw')
TRIPS = RAW_DIR / 'green_tripdata_2025-01.parquet'
ZONES = RAW_DIR / 'taxi_zone_lookup.csv'

assert TRIPS.exists(), f'Не найден файл {TRIPS}'
assert ZONES.exists(), f'Не найден файл {ZONES}'

con = duckdb.connect()

def query_df(sql):
    return con.execute(sql).fetchdf()

TRIPS_SQL = f"read_parquet('{TRIPS.as_posix()}')"
ZONES_SQL = f"read_csv_auto('{ZONES.as_posix()}')"

In [ ]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for path in (TRIPS, ZONES):
    print(f'{path}: {path.stat().st_size:,} bytes; sha256={sha256(path)}')

## 1. Проверяем, что файл содержит нужные поля

На этом шаге подтверждаем техническую пригодность: есть ли время, зоны, расстояние и денежные показатели.

In [ ]:
query_df(f"DESCRIBE SELECT * FROM {TRIPS_SQL}")

In [ ]:
query_df(f"""
    SELECT
        lpep_pickup_datetime,
        lpep_dropoff_datetime,
        PULocationID,
        DOLocationID,
        trip_distance,
        fare_amount,
        tip_amount,
        total_amount,
        payment_type
    FROM {TRIPS_SQL}
    LIMIT 5
""")

## 2. Проверяем фактическое покрытие

Имя файла говорит «2025-01», но это нужно подтвердить по полю времени посадки.

In [ ]:
query_df(f"""
    SELECT
        count(*) AS rows,
        min(lpep_pickup_datetime) AS first_pickup,
        max(lpep_pickup_datetime) AS last_pickup,
        count_if(
            lpep_pickup_datetime < TIMESTAMP '2025-01-01'
            OR lpep_pickup_datetime >= TIMESTAMP '2025-02-01'
        ) AS pickup_outside_january
    FROM {TRIPS_SQL}
""")

## 3. Формулируем первые проверки качества

Это не очистка. Мы только измеряем, какие правила потребуется принять в следующей лабораторной работе. Для каждого правила показаны число строк и его доля от всех поездок.

In [ ]:
query_df(f"""
    SELECT
        count(*) AS total_rows,

        count_if(lpep_dropoff_datetime <= lpep_pickup_datetime) AS non_positive_duration,
        round(100.0 * count_if(lpep_dropoff_datetime <= lpep_pickup_datetime) / count(*), 2) AS non_positive_duration_pct,

        count_if(trip_distance <= 0) AS non_positive_distance,
        round(100.0 * count_if(trip_distance <= 0) / count(*), 2) AS non_positive_distance_pct,

        count_if(total_amount < 0) AS negative_total_amount,
        round(100.0 * count_if(total_amount < 0) / count(*), 2) AS negative_total_amount_pct,

        count_if(payment_type IS NULL) AS missing_payment_type,
        round(100.0 * count_if(payment_type IS NULL) / count(*), 2) AS missing_payment_type_pct
    FROM {TRIPS_SQL}
""")

## 4. Проверяем связь со справочником зон

Справочник нужен, чтобы превратить идентификаторы зон в понятные для анализа районы Нью-Йорка и названия зон.

In [ ]:
query_df(f"""
    SELECT
        count_if(pu.LocationID IS NULL) AS pickup_zone_not_found,
        count_if(dz.LocationID IS NULL) AS dropoff_zone_not_found
    FROM {TRIPS_SQL} AS trip
    LEFT JOIN {ZONES_SQL} AS pu ON trip.PULocationID = pu.LocationID
    LEFT JOIN {ZONES_SQL} AS dz ON trip.DOLocationID = dz.LocationID
""")

## Вывод для ЛР № 1

Набор подходит с ограничениями: нужные поля и справочник зон доступны, но фактический период следует задавать фильтром, а правила обработки некорректных длительностей, расстояний и сумм предстоит определить в дальнейших ЛР. Сейчас ничего не исправляем.